# 1.6 torchao / PyTorch AO 量化工具链

## 为什么要单独讲？

`torchao` 是 PyTorch 官方量化与低精度库，与 **torch.export → ExecuTorch** 路径原生衔接，逐渐成为 PyTorch 端侧量化默认选项（替代零散第三方 PTQ 脚本）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

## 核心能力（概念）

| 能力 | 说明 | 端侧关系 |
|------|------|---------|
| Weight-only INT4/INT8 | 权重量化 API | CPU/GPU/NPU 导出前 |
| Dynamic / Static Act Quant | 激活量化 | ExecuTorch QNN/XNNPACK |
| Float8 | FP8 训练/推理 | 新 GPU，端侧较少 |
| Sparsity | 2:4 等 | NVIDIA 稀疏核 |

未安装 `torchao` 时，用等价伪量化演示 API 形状。

In [ ]:
def weight_only_int4_linear(x, w, group_size=128):
    """模拟 torchao weight-only int4 线性层。"""
    out_f, in_f = w.shape
    assert in_f % group_size == 0
    wg = w.view(out_f, -1, group_size)
    scale = wg.abs().amax(-1, keepdim=True).clamp(min=1e-8) / 7.0
    q = torch.round(wg / scale).clamp(-8, 7)
    w_hat = (q * scale).view_as(w)
    return F.linear(x, w_hat), w_hat


w = torch.randn(256, 512) * 0.02
x = torch.randn(4, 512)
y, w_hat = weight_only_int4_linear(x, w)
err = (F.linear(x, w) - y).norm() / F.linear(x, w).norm()
print(f"INT4 weight-only 相对误差: {err.item():.4f}")
print("推荐流水线: train/finetune → torchao quant → torch.export → ExecuTorch backend")
try:
    import torchao  # noqa: F401
    print("已安装 torchao:", torchao.__version__ if hasattr(torchao, "__version__") else "unknown")
except Exception:
    print("未安装 torchao（可选）: pip install torchao")

## 与 AWQ/GPTQ/bitsandbytes 的分工

- **研究/HuggingFace 生态**：AWQ、GPTQ、bitsandbytes 仍常用。
- **PyTorch 官方端侧**：优先 `torchao` + ExecuTorch。
- **CPU GGUF**：仍以 llama.cpp 量化工具为主。